# 회귀 (Regression) 실습

이번 실습에서는 **선형 회귀** 모델을 직접 학습시켜 보면서, 연속형 변수를 예측하는 방법을 다뤄보겠습니다.

먼저 학생 성적 데이터로 단순/다중 선형 회귀를 익힌 뒤, **중고차 가격 데이터로 직접 처음부터 끝까지 회귀 모델을 만들어 보는 실습**도 함께 진행할 예정입니다.

### 실습 순서
1. 데이터 살펴보기 및 전처리
2. **단순 선형 회귀 (Simple Linear Regression)**
3. **다중 선형 회귀 (Multiple Linear Regression)**
4. **📝 실습 과제: 중고차 가격 예측**

### 사용 라이브러리
- `pandas`, `numpy`: 데이터 처리
- `matplotlib`, `seaborn`: 시각화
- `scikit-learn`: 회귀 모델 학습 및 평가

> 코랩 환경에서는 위 라이브러리들이 기본으로 설치되어 있어 별도 설치는 필요하지 않습니다.
> 만약 환경이 다르다면 아래 셀의 주석을 풀어 설치해 주세요.

In [ ]:
# 필요한 경우에만 실행
# !pip install -q numpy pandas matplotlib seaborn scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 시각화 옵션
plt.rcParams['figure.figsize'] = (8, 5)
sns.set_style('whitegrid')

## 1. 데이터 살펴보기

학생들의 학습 관련 데이터인 `student_score.csv` 파일을 불러오겠습니다.

각 열의 의미는 다음과 같습니다.

| 컬럼명 | 설명 |
| --- | --- |
| `study_hours` | 주당 자기주도 학습 시간 (시간) |
| `sleep_hours` | 일 평균 수면 시간 (시간) |
| `attendance` | 출석률 (%) |
| `prev_score` | 직전 학기 시험 점수 |
| `private_lesson` | 주당 사교육 시간 (시간) |
| `score` | **최종 시험 점수 (예측 대상)** |

> 코랩에서 실습한다면, 제공받은 `student_score.csv` 파일을 코랩의 파일 영역에 업로드한 뒤 아래 코드를 실행해 주세요.

In [ ]:
df = pd.read_csv('student_score.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### 변수 간 상관관계 살펴보기

`score`(최종 점수)와 어떤 변수가 가장 관련이 깊은지 상관계수와 히트맵을 통해 확인해 봅시다.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

히트맵을 살펴보면 `study_hours`와 `score` 사이의 상관계수가 가장 높다는 것을 알 수 있습니다.

이 두 변수의 관계를 산점도로도 한번 확인해 봅시다.

In [ ]:
plt.scatter(df['study_hours'], df['score'], alpha=0.6)
plt.xlabel('study_hours')
plt.ylabel('score')
plt.title('study_hours vs score')
plt.show()

공부 시간이 늘어날수록 점수도 어느 정도 선형적으로 증가하는 경향이 보이네요.

이런 관계를 수치적으로 모델링하는 가장 기본적인 방법이 **선형 회귀**입니다.

## 2. 단순 선형 회귀 (Simple Linear Regression)

먼저 가장 단순한 형태인 **하나의 독립변수만** 사용하는 회귀를 다뤄 보겠습니다.

$$ y = \beta_0 + \beta_1 x + \varepsilon $$

여기서
- $y$: 최종 점수 (`score`)
- $x$: 공부 시간 (`study_hours`)
- $\beta_0$: 절편 (intercept)
- $\beta_1$: 기울기 (slope)
- $\varepsilon$: 오차항

scikit-learn의 `LinearRegression`이 자동으로 $\beta_0, \beta_1$ 값을 **최소제곱법**으로 추정해 줍니다.

### 2-1. 학습/테스트 데이터 분리

모델이 정말 새로운 데이터에서도 잘 작동하는지 확인하려면, **학습에 사용하지 않은 데이터로 평가**해야 합니다.
그래서 우리는 데이터를 두 덩어리로 나누어요.

- **학습 데이터(train)**: 모델이 보고 배우는 데이터 — 보통 전체의 70~80%
- **테스트 데이터(test)**: 모델 평가 전용. 학습 중에는 절대 보여주지 않음 — 보통 20~30%

`sklearn`의 `train_test_split` 함수가 이걸 자동으로 해 줍니다.
`random_state` 를 고정해 두면, 같은 코드를 다시 실행해도 매번 똑같은 분할이 나와서 결과를 재현할 수 있어요.

In [ ]:
# X: 독립변수, y: 종속변수
X = df[['study_hours']]   # ⚠️ 대괄호 두 개! (2차원 DataFrame 으로 만들어야 sklearn 에 들어감)
y = df['score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20%를 테스트용으로
    random_state=42,      # 재현성을 위한 seed
)

print('학습 데이터:', X_train.shape, y_train.shape)
print('테스트 데이터:', X_test.shape, y_test.shape)

### 2-2. 모델 학습

이제 본격적으로 모델을 만들어 학습시켜 봅시다. scikit-learn 에서 모델을 쓰는 흐름은 거의 항상 동일합니다.

1. **객체 생성**: `model = LinearRegression()`
2. **학습 (fit)**: `model.fit(X_train, y_train)` — 데이터를 보고 계수를 찾는 단계
3. **예측 (predict)**: `model.predict(X_new)` — 새 데이터에 대해 예측값을 만드는 단계

`fit()` 한 번 호출로 회귀 계수 $\beta_0$ (절편)와 $\beta_1$ (기울기)이 자동으로 추정돼서 모델 안에 저장됩니다.

In [ ]:
# 1) 객체 생성
slr = LinearRegression()

# 2) 학습 (X_train, y_train 으로 직선 식의 계수를 찾음)
slr.fit(X_train, y_train)

# 3) 학습된 계수 확인
print(f'절편(intercept) β0 = {slr.intercept_:.4f}')
print(f'기울기(coef)    β1 = {slr.coef_[0]:.4f}')

결과를 해석해 보면, 학습된 직선의 식은 대략

$$ \hat{y} = \beta_0 + \beta_1 \cdot \text{study\_hours} $$

가 됩니다. 즉, **공부 시간이 1시간 늘어날 때 점수가 약 $\beta_1$점 정도 오른다**라고 해석할 수 있습니다.

### 2-3. 회귀선 시각화

학습된 모델이 데이터에 잘 들어맞는지 산점도 위에 회귀선을 그려서 직접 눈으로 확인해 봅시다.

In [ ]:
x_line = pd.DataFrame({'study_hours': np.linspace(df['study_hours'].min(), df['study_hours'].max(), 100)})
y_line = slr.predict(x_line)

plt.scatter(X_train, y_train, alpha=0.5, label='train')
plt.scatter(X_test, y_test, alpha=0.7, color='orange', label='test')
plt.plot(x_line, y_line, color='red', linewidth=2, label='regression line')
plt.xlabel('study_hours')
plt.ylabel('score')
plt.title('Simple Linear Regression')
plt.legend()
plt.show()

### 2-4. 모델 평가

회귀 모델의 성능은 보통 아래 지표들로 측정합니다.

- **MSE** (Mean Squared Error): 오차 제곱의 평균. 작을수록 좋음
- **RMSE** (Root Mean Squared Error): MSE의 제곱근. 단위가 y와 같아 해석이 직관적
- **MAE** (Mean Absolute Error): 오차 절댓값의 평균
- **R²** (결정계수): 1에 가까울수록 모델이 데이터를 잘 설명함

In [ ]:
# 학습되지 않았던 X_test 에 대해 예측
y_pred = slr.predict(X_test)

# 실제값 y_test 와 예측값 y_pred 를 비교
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

공부 시간 하나만으로도 약 70% 정도의 점수 변동을 설명할 수 있네요. 꽤 괜찮은 수준이지만, **다른 변수들도 같이 고려한다면 더 정확하게 예측**할 수 있을 거예요.

그래서 이제 변수를 여러 개 동시에 사용하는 **다중 선형 회귀**로 넘어가 봅시다.

## 3. 다중 선형 회귀 (Multiple Linear Regression)

이번에는 여러 개의 독립변수를 동시에 사용해서 점수를 예측해 보겠습니다.

$$ y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p + \varepsilon $$

각 $\beta_i$는 **다른 변수들이 동일할 때**, 해당 변수 $x_i$가 1단위 증가할 때 $y$가 얼마나 변하는지를 의미합니다.

In [ ]:
X = df.drop('score', axis=1)   # score 제외한 모든 컬럼이 독립변수
y = df['score']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('학습 데이터 shape:', X_train.shape)
print('테스트 데이터 shape:', X_test.shape)

In [ ]:
mlr = LinearRegression()
mlr.fit(X_train, y_train)

print(f'intercept: {mlr.intercept_:.4f}')
print('coefficients:')
for name, coef in zip(X.columns, mlr.coef_):
    print(f'  {name:<16s} {coef:.4f}')

### 3-1. 계수 해석

학습된 계수를 보면 각 변수가 점수에 어떤 영향을 주는지 알 수 있습니다.
예를 들어 `study_hours`의 계수가 1.57 이라면, **다른 조건이 같을 때 주당 공부 시간을 1시간 늘리면 점수가 약 1.57점 오른다**는 의미입니다.

각 계수의 크기를 한 번 시각화 해서 비교해 봅시다.

In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coefficient': mlr.coef_
}).sort_values('coefficient', ascending=True)

plt.barh(coef_df['feature'], coef_df['coefficient'])
plt.title('Linear Regression Coefficients')
plt.xlabel('coefficient')
plt.show()

### 3-2. 성능 평가

단순 회귀와 똑같은 방식으로 평가합니다. `predict` 로 예측값을 만들고, 같은 4가지 지표(MSE / RMSE / MAE / R²)를 계산해 비교해 봅시다.

In [ ]:
y_pred = mlr.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'MSE  : {mse:.4f}')
print(f'RMSE : {rmse:.4f}')
print(f'MAE  : {mae:.4f}')
print(f'R²   : {r2:.4f}')

단순 선형 회귀와 비교해 보면 RMSE가 절반 가까이 줄어들고, R²도 0.7대에서 0.9대로 크게 올라간 것을 볼 수 있습니다.

여러 변수를 같이 고려하니 모델이 훨씬 정교해진 거예요.

### 3-3. 예측값 vs 실제값 시각화

산점도로 그렸을 때 점들이 $y = x$ 직선 근처에 모일수록 예측을 잘 한 모델이라고 할 수 있습니다.

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
plt.xlabel('Actual score')
plt.ylabel('Predicted score')
plt.title('Predicted vs Actual')
plt.show()

### 📝 중간 실습: 새로운 학생의 점수 예측하기

아래와 같은 새로운 학생이 한 명 있을 때, 이 학생의 최종 점수를 우리가 학습한 `mlr` 모델로 예측해 봅시다.

| 변수 | 값 |
| --- | --- |
| `study_hours` | 12 |
| `sleep_hours` | 7 |
| `attendance` | 90 |
| `prev_score` | 75 |
| `private_lesson` | 3 |

**HINT**: scikit-learn 모델은 2차원 형태의 입력을 받습니다.
DataFrame이나 `np.array([[...]])` 형태로 만들어 주세요.

In [ ]:
new_student = pd.DataFrame([{
    'study_hours': 12,
    'sleep_hours': 7,
    'attendance': 90,
    'prev_score': 75,
    'private_lesson': 3,
}])

predicted = mlr.predict(new_student)
print(f'예상 점수: {predicted[0]:.2f}점')

회귀 분석의 전체 흐름을 정리하면 이렇게 됩니다.

1. **데이터 살펴보기** (`head`, `info`, `describe`, 상관관계)
2. **train / test 분리** (`train_test_split`)
3. **모델 학습** (`LinearRegression()` → `fit`)
4. **계수 확인 및 해석** (`intercept_`, `coef_`)
5. **예측 + 평가** (`predict` → RMSE, R²)
6. **새 입력값으로 실제 예측**

다음 섹션에서는 이 흐름 그대로 새로운 데이터(중고차 가격) 에 적용해 보겠습니다.
앞에서 본 코드를 참고하면서 직접 따라 작성해 보세요!

## 4. 📝 실습 과제: 중고차 가격 예측

이번에는 학생 데이터에서 배운 회귀 분석 절차를 **여러분이 직접** 처음부터 적용해 볼 차례입니다.

`used_car.csv` 파일에는 중고차 220대의 정보가 들어 있고, 우리는 이 데이터로 **중고차 가격(만원)을 예측하는 회귀 모델**을 만들어 볼 거예요.

| 컬럼명 | 설명 |
| --- | --- |
| `mileage_km` | 누적 주행거리 (km) |
| `car_age` | 차량 연식 (년) |
| `engine_cc` | 배기량 (cc) |
| `fuel_efficiency` | 공인 연비 (km/L) |
| `accident_count` | 사고 이력 건수 |
| `price_man` | **중고차 가격 (만원, 예측 대상)** |

진행 순서는 학생 데이터 예제와 **완전히 똑같습니다**.

1. 데이터 불러오고 살펴보기
2. 변수 간 상관관계 확인
3. 단순 선형 회귀 (가장 상관이 강한 변수 하나로)
4. 다중 선형 회귀 (모든 변수 사용)
5. 직접 입력값을 넣어 가격 예측해 보기


> 코랩에서 실습한다면, 제공받은 `used_car.csv` 파일도 코랩의 파일 영역에 같이 업로드해 주세요.

### 4-1. 데이터 불러오고 살펴보기

`used_car.csv` 를 읽어와서 `car_df` 라는 변수에 저장하고, 첫 5행을 출력해 보세요.

데이터의 크기, 결측치, 통계량을 확인해 봅시다.

### 4-2. 변수 간 상관관계 확인

`price_man` (가격)과 가장 강한 상관관계를 가지는 변수가 무엇인지 히트맵으로 확인해 봅시다.

상관계수 표를 직접 출력해서 `price_man` 과 가장 강한 (음/양의) 상관관계를 가지는 변수가 무엇인지도 확인해 봅시다.

가격과 가장 강한 상관관계(절댓값 기준)를 가지는 변수는 **X**입니다.
연식이 오래될수록 가격이 떨어지는 자연스러운 관계가 보이네요.

`X` 와 `price_man` 의 산점도를 그려서 직접 눈으로 확인해 봅시다.

### 4-3. 단순 선형 회귀: `X` → `price_man`

이번에는 `X` 하나만 가지고 가격을 예측하는 단순 회귀 모델을 만들어 보겠습니다.

학습/테스트 데이터 분리부터 학습, 평가까지 직접 진행해 보세요.

모델이 학습한 직선을 산점도 위에 같이 그려서 확인해 봅시다.

테스트 세트에서의 성능(RMSE, R²) 도 측정해 봅시다.

변수 하나만으로도 어느 정도 가격을 설명할 수 있다는 걸 확인할 수 있습니다.
하지만 R² 가 그렇게 높지는 않죠. 다른 변수들도 같이 써서 모델을 더 정교하게 만들어 봅시다.

### 4-4. 다중 선형 회귀: 모든 변수 사용

이번에는 `price_man` 을 제외한 모든 변수를 사용해 다중 선형 회귀를 학습해 봅시다.

학습한 모델의 RMSE 와 R² 를 측정하고, 단순 회귀와 비교해 봅시다.

단순 회귀에 비해 RMSE 가 크게 줄고, R² 가 0.9 이상으로 올라간 것을 볼 수 있습니다.

여러 변수를 함께 보는 다중 회귀가 훨씬 더 정확한 예측을 한다는 것을 확인할 수 있어요.

### 📊 (참고) 각 변수의 계수 시각화

각 변수가 가격에 어떻게 영향을 주는지 막대 그래프로 확인해 봅시다.
- 계수가 **양수**이면 그 변수 값이 커질 때 가격이 **오르고**,
- **음수**이면 가격이 **떨어지는** 방향이라는 의미입니다.

### 4-5. 🚗 내가 사고 싶은 차의 가격 예측해 보기

이제 학습된 `car_mlr` 모델로 **여러분이 직접 입력한 중고차 정보**에 대해 가격을 예측해 봅시다.

아래 셀의 값을 자유롭게 바꾸면서 가격이 어떻게 변하는지 확인해 보세요!

예시:
- 5년 된 2,000cc 차량, 7만 km 주행, 연비 13, 사고 0건
- 12년 된 1,000cc 경차, 18만 km 주행, 연비 16, 사고 2건

In [ ]:
# 🎯 my_car 의 값들을 자유롭게 바꿔 보세요!
my_car = pd.DataFrame([{
    'mileage_km': 70000,
    'car_age': 5,
    'engine_cc': 2000,
    'fuel_efficiency': 13.0,
    'accident_count': 0,
}])




수고하셨습니다! 🎉